## Импорты

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import warnings
warnings.filterwarnings('ignore')

Настройка лемматизации и стоп-слов

In [ ]:
try:
    import pymorphy3
    morph = pymorphy3.MorphAnalyzer()
    lemmatize = True
    KEEP_POS = {'NOUN', 'ADJF', 'ADJS', 'VERB', 'INFN', 'PRTF', 'PRTS', 'GRND'}
except ImportError:
    lemmatize = False
    KEEP_POS = None

RUSSIAN_STOPWORDS = set('а,без,более,бы,был,была,были,было,быть,в,вам,вас,весь,во,вот,все,всего,всех,вы,где,да,даже,для,до,его,ее,если,есть,еще,же,за,здесь,и,из,или,им,их,к,как,ко,когда,кто,ли,либо,мне,может,мы,на,над,не,него,нее,нет,ни,них,но,ну,о,об,однако,он,она,они,оно,от,очень,по,под,при,с,со,так,также,такой,там,те,тем,то,того,тоже,той,только,том,ты,у,уже,хотя,чего,чей,чем,что,чтобы,чье,чья,эта,эти,это,этот,я'.split(','))
RUSSIAN_STOPWORDS = {word.strip() for word in RUSSIAN_STOPWORDS}

Загрузка и предобработка данных

In [ ]:
df = pd.read_csv("all_banks_reviews.csv", sep=';')
df['rating'] = pd.to_numeric(df['rating'].astype(str).str.replace(',', '.'), errors='coerce')

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^а-яa-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def lemmatize_and_filter(text):
    if not lemmatize or not text.strip():
        return text
    words = text.split()
    lemmas = []
    for w in words:
        p = morph.parse(w)[0]
        if p.tag.POS in KEEP_POS:
            lemma = p.normal_form
            if lemma not in RUSSIAN_STOPWORDS and len(lemma) > 2:
                lemmas.append(lemma)
    return ' '.join(lemmas)

df['pros_clean'] = df['pros'].fillna('').apply(clean_text).apply(lemmatize_and_filter)
df['cons_clean'] = df['cons'].fillna('').apply(clean_text).apply(lemmatize_and_filter)
df['text'] = df['pros_clean'] + ' ' + df['cons_clean']

LLM анализ отзывов

In [ ]:
import openai
import json
import time

openai.api_key = "YOUR_OPENAI_API_KEY"

def analyze_review_with_llm(pros, cons):
    prompt = f"""
    Проанализируй отзыв сотрудника о компании.
    Плюсы: {pros}
    Минусы: {cons}

    Ответь строго в формате JSON. Допустимые значения для sentiment: "positive", "neutral", "negative".
    {{
        "sentiment": "positive/neutral/negative",
        "topics_pros": ["тема1", "тема2"],
        "topics_cons": ["тема1", "тема2"],
        "explanation": "краткое пояснение"
    }}
    """
    try:
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=500,
            response_format={"type": "json_object"}
        )
        data = json.loads(response.choices[0].message.content)
        sentiment = data.get('sentiment', 'unknown').lower()
        if sentiment not in ['positive', 'neutral', 'negative']:
            sentiment = 'unknown'
        data['sentiment'] = sentiment
        return data
    except:
        return {"sentiment": "unknown", "topics_pros": [], "topics_cons": [], "explanation": ""}

df['llm_sentiment'] = None
df['llm_topics_pros'] = None
df['llm_topics_cons'] = None

batch_size = 100
output_file = 'reviews_llm_analysis.csv'

for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size].copy()
    for idx, row in batch.iterrows():
        result = analyze_review_with_llm(row['pros'] or '', row['cons'] or '')
        df.at[idx, 'llm_sentiment'] = result.get('sentiment')
        df.at[idx, 'llm_topics_pros'] = str(result.get('topics_pros', []))
        df.at[idx, 'llm_topics_cons'] = str(result.get('topics_cons', []))
        time.sleep(0.2)
    df.to_csv(output_file, index=False, encoding='utf-8-sig')